# INF2006 G12 - Task 1: raw corpus inventory, provenance and quality (EMR, PySpark)

Profiles the 5 staged sources in `/corpus/raw` (read-only): **prices, SEC EDGAR, SEC inventory, GDELT, Wikinews**.
Every check prints a small table and saves it as JSON under
`/corpus/raw/_manifests/<source>/<version>/acq_<date>/profile/<RUN_ID>/`.

**Before the first run** (once, in an SSH shell on the primary node). The notebook user `jovyan` cannot write there by default:
```bash
sudo -u hdfs hdfs dfs -mkdir -p /user/jovyan && sudo -u hdfs hdfs dfs -chown jovyan /user/jovyan
hdfs dfs -chmod -R 777 /corpus/raw/_manifests
```

**How to run:** kernel = **PySpark**. Run the cells in order.
`FULL_RUN = False` (cell 1) is a quick dry run of GDELT (5 files) and Wikinews (first 3M lines).
Set `FULL_RUN = True` for the evidence run (GDELT about 20-40 min, Wikinews about 10 min), then export the notebook.

**Screenshots (only these):** cells 1, 3, 4, 7, 12 and 15. Cell 10 is optional.

## 1. Setup and cluster configuration
**Screenshot:** the settings table (EMR/Spark/HDFS configuration).

In [1]:
# [1] Setup: parameters, Spark session, cluster configuration
import datetime, platform, time
from contextlib import contextmanager
from pyspark.sql import functions as F

try:
    spark                                   # predefined in Livy and Zeppelin sessions
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

FULL_RUN = False          # False = quick dry run of GDELT (5 files) and Wikinews (3M lines). True = full evidence run.
GDELT_LIMIT = None if FULL_RUN else 5
WIKI_LINE_LIMIT = None if FULL_RUN else 3000000

S3_BUCKET = "inf2006-bigdata"
SRC = {   # source: (version, acquisition date) -> /corpus/raw/<source>/<version>/acq_<date>/
    "prices":        ("v1_2013-2017",          "2026-10-28"),
    "sec_inventory": ("v1",                    "2026-10-28"),
    "sec_edgar":     ("v1",                    "2026-10-28"),
    "wikinews":      ("20261001_meta-history", "2026-10-29"),
    "gdelt":         ("v1_events_2015-2017",   "2026-10-29"),
}
EXPECTED_FILES = {"prices": 1, "sec_inventory": 1, "sec_edgar": 311, "wikinews": 2, "gdelt": 731}
RAW = lambda s: f"/corpus/raw/{s}/{SRC[s][0]}/acq_{SRC[s][1]}"
MAN = lambda s: f"/corpus/raw/_manifests/{s}/{SRC[s][0]}/acq_{SRC[s][1]}"
PROFILE_ROOT = None      # None = save results next to each source's manifests
RUN_ID = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
TIMINGS = {}

_jvm, _hconf = sc._jvm, sc._jsc.hadoopConfiguration()
_Path = _jvm.org.apache.hadoop.fs.Path
def _fs(p): return _Path(p).getFileSystem(_hconf)
def fs_exists(p): return _fs(p).exists(_Path(p))
def fs_list(p):
    it, out = _fs(p).listFiles(_Path(p), True), []
    while it.hasNext():
        st = it.next()
        out.append((st.getPath().toString(), st.getLen(), st.getReplication()))
    return sorted(out)

def out_dir(src):
    if src == "_run":
        base = f"{PROFILE_ROOT}/_run" if PROFILE_ROOT else "/corpus/raw/_manifests/_task1_run"
    else:
        base = f"{PROFILE_ROOT}/{src}" if PROFILE_ROOT else f"{MAN(src)}/profile"
    return f"{base}/{RUN_ID}"

def save(df, src, name, rows=60):
    """Write a small result as JSON (the evidence file) and show it (the screenshot)."""
    path = f"{out_dir(src)}/{name}"
    df.coalesce(1).write.mode("overwrite").json(path)
    print(f"[{src}] {name}  ->  {path}")
    df.show(rows, truncate=False)

@contextmanager
def timed(name):
    t = time.time(); yield
    TIMINGS[name] = round(time.time() - t, 1); print(f"-- {name}: {TIMINGS[name]} s")

conf = dict(sc.getConf().getAll())
try: live_exec = str(sc._jsc.sc().getExecutorMemoryStatus().size() - 1)
except Exception: live_exec = "n/a"
rows = [("run_id", RUN_ID), ("full_run", str(FULL_RUN)), ("application_id", sc.applicationId),
        ("spark.version", spark.version), ("hadoop.version", _jvm.org.apache.hadoop.util.VersionInfo.getVersion()),
        ("python", platform.python_version()), ("dfs.replication", _hconf.get("dfs.replication")),
        ("executors (live)", live_exec)] + \
       [(k, conf.get(k, "(default)")) for k in ["spark.master", "spark.executor.instances", "spark.executor.cores",
        "spark.executor.memory", "spark.executor.memoryOverhead", "spark.driver.memory",
        "spark.dynamicAllocation.enabled"]]
save(spark.createDataFrame(rows, "setting string, value string"), "_run", "00_cluster_config")

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
4,application_1793529609230_0007,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[_run] 00_cluster_config  ->  /corpus/raw/_manifests/_task1_run/20261101T112713Z/00_cluster_config
+-------------------------------+------------------------------+
|setting                        |value                         |
+-------------------------------+------------------------------+
|run_id                         |20261101T112713Z              |
|full_run                       |False                         |
|application_id                 |application_1793529609230_0007|
|spark.version                  |3.5.5-amzn-1                  |
|hadoop.version                 |3.4.1-amzn-2                  |
|python                         |3.9.25                        |
|dfs.replication                |2                             |
|executors (live)               |1                             |
|spark.master                   |yarn                          |
|spark.executor.instances       |2                             |
|spark.executor.cores           |2                      

## 2. Provenance manifests
Copies the original download manifests (retrieval times, URLs, sizes, checksums) from S3 into HDFS `_manifests`. No screenshot.

In [2]:
# [2] Copy the original download manifests from S3 into HDFS _manifests
PROV_S3 = f"s3://{S3_BUCKET}/evidence/download_manifests"
FileUtil = _jvm.org.apache.hadoop.fs.FileUtil
copied = []
for s in SRC:
    d = f"{PROV_S3}/{s}"
    if not fs_exists(d):
        copied.append((s, "(none)", 0, "S3 folder missing")); continue
    _fs(MAN(s)).mkdirs(_Path(MAN(s)))
    for st in _fs(d).listStatus(_Path(d)):
        name = st.getPath().getName()
        ok = FileUtil.copy(_fs(d), st.getPath(), _fs(MAN(s)), _Path(f"{MAN(s)}/{name}"), False, True, _hconf)
        copied.append((s, name, st.getLen(), "copied" if ok else "FAILED"))
save(spark.createDataFrame(copied, "source string, file string, bytes long, result string"), "_run", "01_manifests_copied")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[_run] 01_manifests_copied  ->  /corpus/raw/_manifests/_task1_run/20261101T112713Z/01_manifests_copied
+-------------+---------------------------+------+------+
|source       |file                       |bytes |result|
+-------------+---------------------------+------+------+
|prices       |acquisition_manifest.csv   |1214  |copied|
|sec_inventory|inventory_sha256_laptop.txt|92    |copied|
|sec_edgar    |sec_acquisition_summary.txt|630   |copied|
|sec_edgar    |sec_download_manifest.csv  |129442|copied|
|sec_edgar    |sec_inventory_checks.csv   |35    |copied|
|sec_edgar    |sec_unavailable.csv        |208   |copied|
|wikinews     |acquisition_summary.txt    |370   |copied|
|wikinews     |download_manifest.csv      |3100  |copied|
|wikinews     |wikinews_dumpstatus.json   |22245 |copied|
|gdelt        |acquisition_summary.txt    |374   |copied|
|gdelt        |download_manifest.csv      |289025|copied|
|gdelt        |gdelt_filesizes.txt        |162079|copied|
|gdelt        |gdelt_md5sum

## 3. Corpus inventory
**Screenshot:** the `02_inventory` table and the `TOTAL` line (1,046 files, replication 2).

In [3]:
# [3] Corpus inventory: what is in HDFS right now, and the staging self-checks
rows = []
for s in SRC:
    f = fs_list(RAW(s))
    rows.append((s, SRC[s][0], SRC[s][1], EXPECTED_FILES[s], len(f), sum(x[1] for x in f),
                 min(x[2] for x in f) if f else None, max(x[2] for x in f) if f else None, RAW(s)))
inv = spark.createDataFrame(rows, "source string, version string, acq_date string, expected_files int, "
                                  "hdfs_files int, hdfs_bytes long, min_repl int, max_repl int, hdfs_path string")
inv = inv.withColumn("files_ok", F.col("hdfs_files") == F.col("expected_files"))
save(inv, "_run", "02_inventory")
t_ = inv.agg(F.sum("hdfs_files").alias("files"), F.sum("hdfs_bytes").alias("bytes")).collect()[0]
print(f"TOTAL: {t_.files} files, {t_.bytes:,} bytes ({t_.bytes/1e9:.2f} GB); expected 1,046 files")

rec = None
for s in SRC:
    d = (spark.read.option("header", True).csv(f"{MAN(s)}/staging_record.csv")
         .select(F.lit(s).alias("source"), F.regexp_extract("hdfs_path", r"([^/]+)$", 1).alias("file"),
                 F.col("hdfs_bytes").cast("long").alias("hdfs_bytes"), "local_sha256", "hdfs_sha256", "match", "staged_at_utc"))
    rec = d if rec is None else rec.unionByName(d)
rec = rec.cache()
save(rec.groupBy("source").agg(F.count("*").alias("records"), F.sum("hdfs_bytes").alias("bytes"),
        F.sum(F.when(F.col("local_sha256") == F.col("hdfs_sha256"), 1).otherwise(0)).alias("sha256_equal"),
        F.sum(F.when(F.col("match") == "yes", 1).otherwise(0)).alias("match_yes"),
        F.min("staged_at_utc").alias("first_staged_utc"), F.max("staged_at_utc").alias("last_staged_utc")
     ).orderBy("source"), "_run", "03_staging_self_check")

lines = [(s, ln) for s in SRC if fs_exists(f"{MAN(s)}/staging_summary.txt")
         for ln in sc.textFile(f"{MAN(s)}/staging_summary.txt").collect()]
save(spark.createDataFrame(lines, "source string, staging_summary string"), "_run", "04_staging_summaries", rows=80)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[_run] 02_inventory  ->  /corpus/raw/_manifests/_task1_run/20261101T112713Z/02_inventory
+-------------+---------------------+----------+--------------+----------+----------+--------+--------+---------------------------------------------------------+--------+
|source       |version              |acq_date  |expected_files|hdfs_files|hdfs_bytes|min_repl|max_repl|hdfs_path                                                |files_ok|
+-------------+---------------------+----------+--------------+----------+----------+--------+--------+---------------------------------------------------------+--------+
|prices       |v1_2013-2017         |2026-10-28|1             |1         |29580549  |2       |2       |/corpus/raw/prices/v1_2013-2017/acq_2026-10-28           |true    |
|sec_inventory|v1                   |2026-10-28|1             |1         |51751     |2       |2       |/corpus/raw/sec_inventory/v1/acq_2026-10-28              |true    |
|sec_edgar    |v1                   |2026-10-28|311     

## 4. Provenance chain: original download, S3 copy, HDFS
**Screenshot:** the `06_provenance_chain` table and the `True` line below it.

In [4]:
# [4] Provenance chain: original download (2026-10-28/29) -> S3 copy -> HDFS
def orig_manifest(s):
    m = MAN(s)
    rd = lambda name: spark.read.option("header", True).csv(f"{m}/{name}")
    if s == "prices":
        allrows = rd("acquisition_manifest.csv")
        extra = allrows.filter(F.col("local_file") != "all_stocks_5yr.csv").count()
        if extra: print(f"prices manifest: {extra} row(s) for a file outside the corpus were ignored")
        d = allrows.filter(F.col("local_file") == "all_stocks_5yr.csv").select(
            F.col("local_file").alias("file"), F.col("pinned_url").alias("source_url"),
            "bytes", "sha256", "retrieved_at_utc", "status")
    elif s == "sec_edgar":
        d = rd("sec_download_manifest.csv").select(F.col("local_file").alias("file"), "source_url",
                                                   "bytes", "sha256", "retrieved_at_utc", "status")
    elif s == "sec_inventory":
        t = spark.read.text(f"{m}/inventory_sha256_laptop.txt")
        d = t.select(F.regexp_extract("value", r"^\s*[0-9a-fA-F]{64}\s+\*?(.+?)\s*$", 1).alias("file"),
                     F.lit("course page (CSV twin of the official .xlsx)").alias("source_url"),
                     F.lit(None).cast("string").alias("bytes"),
                     F.regexp_extract("value", r"^\s*([0-9a-fA-F]{64})", 1).alias("sha256"),
                     F.lit(None).cast("string").alias("retrieved_at_utc"), F.lit("ok").alias("status"))
    else:
        d = rd("download_manifest.csv").select("file", "source_url", "bytes", "sha256", "retrieved_at_utc", "status")
    return d.select(F.lit(s).alias("source"), "file", "source_url", F.col("bytes").cast("long").alias("bytes"),
                    "sha256", "retrieved_at_utc", "status")

orig = None
for s in SRC:
    d = orig_manifest(s)
    orig = d if orig is None else orig.unionByName(d)
orig = orig.cache()

host = F.regexp_extract("source_url", r"^https?://([^/]+)", 1)
save(orig.withColumn("host", F.when(host == "", F.col("source_url")).otherwise(host))
         .groupBy("source", "host", "status")
         .agg(F.count("*").alias("attempts"), F.min("retrieved_at_utc").alias("first_utc"),
              F.max("retrieved_at_utc").alias("last_utc")).orderBy("source", "host", "status"),
     "_run", "05_download_attempts")

orig_ok = orig.filter(F.col("status") == "ok").dropDuplicates(["source", "file", "sha256"])
j = rec.join(orig_ok, ["source", "file"], "full_outer")
chain = (j.groupBy("source").agg(
    F.sum(F.when(F.col("hdfs_sha256").isNotNull(), 1).otherwise(0)).alias("hdfs_files"),
    F.sum(F.when(F.col("sha256").isNotNull(), 1).otherwise(0)).alias("original_files"),
    F.sum(F.when(F.col("hdfs_sha256") == F.col("sha256"), 1).otherwise(0)).alias("sha256_identical"),
    F.sum(F.when(F.col("bytes").isNotNull(), 1).otherwise(0)).alias("bytes_in_original"),
    F.sum(F.when(F.col("hdfs_bytes") == F.col("bytes"), 1).otherwise(0)).alias("bytes_identical"),
    F.sum(F.when(F.col("sha256").isNull() & F.col("hdfs_sha256").isNotNull(), 1).otherwise(0)).alias("hdfs_not_in_original"),
    F.sum(F.when(F.col("hdfs_sha256").isNull() & F.col("sha256").isNotNull(), 1).otherwise(0)).alias("original_not_in_hdfs"),
).orderBy("source"))
save(chain, "_run", "06_provenance_chain")
c = chain.collect()
print("Every HDFS file is byte-identical to its original download:",
      all(r.sha256_identical == r.hdfs_files == r.original_files for r in c))

pub = None
for s in ("gdelt", "wikinews"):
    d = (spark.read.option("header", True).csv(f"{MAN(s)}/download_manifest.csv").filter("status = 'ok'")
         .groupBy("source", "expected_hash_algo")
         .agg(F.count("*").alias("files"),
              F.sum(F.when(F.col("hash_match") == "True", 1).otherwise(0)).alias("publisher_hash_matched")))
    pub = d if pub is None else pub.unionByName(d)
save(pub, "_run", "07_publisher_checksums")
print("Prices (pinned GitHub commit) and SEC (no publisher checksum): identity rests on sha256 + a second independent download.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

prices manifest: 1 row(s) for a file outside the corpus were ignored
[_run] 05_download_attempts  ->  /corpus/raw/_manifests/_task1_run/20261101T112713Z/05_download_attempts
+-------------+--------------------------------------------+-----------+--------+--------------------+--------------------+
|source       |host                                        |status     |attempts|first_utc           |last_utc            |
+-------------+--------------------------------------------+-----------+--------+--------------------+--------------------+
|gdelt        |data.gdeltproject.org                       |ok         |731     |2026-10-29T08:29:33Z|2026-10-29T09:04:35Z|
|prices       |raw.githubusercontent.com                   |ok         |1       |2026-10-28T06:05:40Z|2026-10-28T06:05:40Z|
|sec_edgar    |www.sec.gov                                 |ok         |311     |2026-10-28T06:06:40Z|2026-10-28T06:09:48Z|
|sec_inventory|course page (CSV twin of the official .xlsx)|ok         |1       |N

## 5. Prices
Cells 5-8. Only cell 7 needs a screenshot: the `06_reconciliation` table (605,887 accepted bars).

In [5]:
# [5] Prices: read every column as STRING, count lines vs parsed rows
from pyspark.sql import Window as W
from pyspark.sql.types import StructType, StructField, StringType

cnt = lambda cond, name: F.sum(F.when(cond, 1).otherwise(0)).alias(name)
def save_kv(df, src, name):
    """Single-row aggregate -> two-column table (easier to read and screenshot)."""
    r = df.first().asDict()
    save(spark.createDataFrame([(k, "" if v is None else str(v)) for k, v in r.items()],
                               "check string, value string"), src, name, rows=100)

PRICES_RAW = RAW("prices")
EXPECTED_FIRST, EXPECTED_LAST = "2013-02-08", "2017-12-29"     # from the project spec
lines = spark.read.text(PRICES_RAW)
header_line = lines.first()["value"]
cols = [c.strip() for c in header_line.split(",")]
print("header:", cols)
assert {"date", "open", "high", "low", "close", "volume", "Name"} <= set(cols), cols
schema = StructType([StructField(c, StringType()) for c in cols] + [StructField("_corrupt_record", StringType())])
raw = (spark.read.option("header", True).option("mode", "PERMISSIVE")
       .option("columnNameOfCorruptRecord", "_corrupt_record").schema(schema).csv(PRICES_RAW).cache())
save_kv(spark.createDataFrame([(lines.count(), lines.filter(F.col("value") == header_line).count(),
        lines.filter(F.trim("value") == "").count(), raw.count(),
        raw.filter(F.col("_corrupt_record").isNotNull()).count())],
        "physical_lines long, header_lines long, blank_lines long, parsed_rows long, malformed_rows long"),
        "prices", "01_row_counts")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

header: ['date', 'open', 'high', 'low', 'close', 'volume', 'Name']
[prices] 01_row_counts  ->  /corpus/raw/_manifests/prices/v1_2013-2017/acq_2026-10-28/profile/20261101T112713Z/01_row_counts
+--------------+------+
|check         |value |
+--------------+------+
|physical_lines|619041|
|header_lines  |1     |
|blank_lines   |0     |
|parsed_rows   |619040|
|malformed_rows|0     |
+--------------+------+

In [6]:
# [6] Prices: nulls, parse failures, coverage, duplicates
NUM = ["open", "high", "low", "close", "volume"]
save_kv(raw.select([cnt(F.col(c).isNull() | (F.trim(F.col(c)) == ""), c) for c in cols]),
        "prices", "02_null_or_empty_per_column")

t = raw.select(F.col("Name").alias("series"), F.col("date").alias("date_raw"),
               F.to_date("date", "yyyy-MM-dd").alias("date"),
               *[F.col(k).alias(f"{k}_raw") for k in NUM],
               *[F.col(k).cast("double").alias(k) for k in NUM]).cache()
save_kv(t.select(
    cnt(F.col("date_raw").isNotNull() & F.col("date").isNull(), "bad_date"),
    *[cnt(F.col(f"{k}_raw").isNotNull() & (F.trim(F.col(f"{k}_raw")) != "") & F.col(k).isNull(), f"bad_{k}") for k in NUM],
    F.min("date").cast("string").alias("min_date"), F.max("date").cast("string").alias("max_date"),
    F.countDistinct("series").alias("distinct_series"), F.countDistinct("date").alias("distinct_dates")),
    "prices", "03_parse_failures_and_coverage")

key = ["series", "date_raw"]
save_kv(spark.createDataFrame([(
    raw.count() - raw.drop("_corrupt_record").distinct().count(),
    t.groupBy(key).count().filter("count > 1").count(),
    t.groupBy(key).agg(F.countDistinct(F.concat_ws("|", *[F.coalesce(F.col(f"{k}_raw"), F.lit("<null>")) for k in NUM])).alias("v"))
     .filter("v > 1").count())],
    "exact_duplicate_rows long, duplicate_keys long, conflicting_duplicate_keys long"), "prices", "04_duplicates")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[prices] 02_null_or_empty_per_column  ->  /corpus/raw/_manifests/prices/v1_2013-2017/acq_2026-10-28/profile/20261101T112713Z/02_null_or_empty_per_column
+------+-----+
|check |value|
+------+-----+
|date  |0    |
|open  |11   |
|high  |8    |
|low   |8    |
|close |0    |
|volume|0    |
|Name  |0    |
+------+-----+

[prices] 03_parse_failures_and_coverage  ->  /corpus/raw/_manifests/prices/v1_2013-2017/acq_2026-10-28/profile/20261101T112713Z/03_parse_failures_and_coverage
+---------------+----------+
|check          |value     |
+---------------+----------+
|bad_date       |0         |
|bad_open       |0         |
|bad_high       |0         |
|bad_low        |0         |
|bad_close      |0         |
|bad_volume     |0         |
|min_date       |2013-02-08|
|max_date       |2018-02-07|
|distinct_series|505       |
|distinct_dates |1259      |
+---------------+----------+

[prices] 04_duplicates  ->  /corpus/raw/_manifests/prices/v1_2013-2017/acq_2026-10-28/profile/20261101T112713Z/04_d

In [7]:
# [7] Prices: validity checks and reconciliation to the spec's 605,887 accepted bars   (SCREENSHOT: 06_reconciliation)
p = t.filter(F.col("date").isNotNull())
save_kv(p.select(
    cnt((F.col("open") <= 0) | (F.col("high") <= 0) | (F.col("low") <= 0) | (F.col("close") <= 0), "nonpositive_price"),
    cnt(F.col("high") < F.col("low"), "high_lt_low"),
    cnt((F.col("open") > F.col("high")) | (F.col("open") < F.col("low")), "open_outside_range"),
    cnt((F.col("close") > F.col("high")) | (F.col("close") < F.col("low")), "close_outside_range"),
    cnt(F.col("volume") < 0, "negative_volume"), cnt(F.col("volume") == 0, "zero_volume"),
    cnt(F.col("volume") != F.floor("volume"), "fractional_volume")), "prices", "05_price_volume_validity")

reason = (F.when(F.col("open").isNull() | F.col("high").isNull() | F.col("low").isNull()
                 | F.col("close").isNull() | F.col("volume").isNull(), "null_field")
           .when(F.col("high") < F.col("low"), "high_lt_low")
           .when((F.col("open") > F.col("high")) | (F.col("open") < F.col("low")), "open_outside_range")
           .when((F.col("close") > F.col("high")) | (F.col("close") < F.col("low")), "close_outside_range")
           .when(F.col("volume") == 0, "zero_volume"))
scoped = (p.withColumn("in_scope", F.col("date") <= F.lit(EXPECTED_LAST).cast("date"))
           .withColumn("flag", reason)).cache()
save_kv(scoped.agg(F.count("*").alias("raw_rows"),
    cnt(~F.col("in_scope"), "after_2017_12_29_out_of_scope"),
    cnt(F.col("in_scope") & F.col("flag").isNotNull(), "in_scope_flagged"),
    cnt(F.col("in_scope") & F.col("flag").isNull(), "accepted_bars"),
    F.lit(605887).alias("spec_accepted_bars")), "prices", "06_reconciliation")
save(scoped.filter(F.col("in_scope") & F.col("flag").isNotNull()).groupBy("flag").count().orderBy("flag"),
     "prices", "07_flag_reasons")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[prices] 05_price_volume_validity  ->  /corpus/raw/_manifests/prices/v1_2013-2017/acq_2026-10-28/profile/20261101T112713Z/05_price_volume_validity
+-------------------+-----+
|check              |value|
+-------------------+-----+
|nonpositive_price  |0    |
|high_lt_low        |1    |
|open_outside_range |4    |
|close_outside_range|9    |
|negative_volume    |0    |
|zero_volume        |4    |
|fractional_volume  |0    |
+-------------------+-----+

[prices] 06_reconciliation  ->  /corpus/raw/_manifests/prices/v1_2013-2017/acq_2026-10-28/profile/20261101T112713Z/06_reconciliation
+-----------------------------+------+
|check                        |value |
+-----------------------------+------+
|raw_rows                     |619040|
|after_2017_12_29_out_of_scope|13130 |
|in_scope_flagged             |23    |
|accepted_bars                |605887|
|spec_accepted_bars           |605887|
+-----------------------------+------+

[prices] 07_flag_reasons  ->  /corpus/raw/_manifests/prices

In [8]:
# [8] Prices: extreme day-over-day moves and per-series coverage (counted, nothing corrected)
w = W.partitionBy("series").orderBy("date")
moves = p.withColumn("prev_close", F.lag("close").over(w)).withColumn("ret", F.col("close") / F.col("prev_close") - 1)
save_kv(moves.select(cnt(F.abs("ret") > 0.5, "abs_move_gt_50pct"), cnt(F.abs("ret") > 0.2, "abs_move_gt_20pct"),
        F.expr("percentile_approx(ret, array(0.001,0.01,0.5,0.99,0.999))").alias("ret_quantiles_p01_p1_p50_p99_p999")),
        "prices", "08_extreme_moves_summary")
save(moves.filter(F.abs("ret") > 0.5)
          .select("series", F.col("date").cast("string").alias("date"), "prev_close", "close", F.round("ret", 3).alias("ret"))
          .orderBy(F.abs("ret").desc()), "prices", "09_moves_over_50pct")

cal = p.select("date").distinct().withColumn("cal_idx", F.row_number().over(W.orderBy("date")))
per = (p.select("series", "date").distinct().join(cal, "date").groupBy("series")
        .agg(F.count("*").alias("bars"), F.min("date").alias("first_date"), F.max("date").alias("last_date"),
             (F.max("cal_idx") - F.min("cal_idx") + 1).alias("calendar_days_in_span"))
        .withColumn("missing_days_in_span", F.col("calendar_days_in_span") - F.col("bars"))
        .withColumn("starts_late", F.col("first_date") > F.lit(EXPECTED_FIRST).cast("date"))
        .withColumn("ends_early", F.col("last_date") < F.lit(EXPECTED_LAST).cast("date"))).cache()
save_kv(per.agg(F.count("*").alias("series"), cnt(F.col("starts_late"), "series_starting_late"),
        cnt(F.col("ends_early"), "series_ending_early"), cnt(F.col("missing_days_in_span") > 0, "series_with_gaps"),
        F.sum("missing_days_in_span").alias("missing_bars_in_span"), F.min("bars").alias("min_bars"),
        F.max("bars").alias("max_bars"), F.expr("percentile_approx(bars, 0.5)").alias("median_bars")),
        "prices", "10_per_series_summary")
save(per.orderBy("bars").select("series", "bars", F.col("first_date").cast("string").alias("first_date"),
        F.col("last_date").cast("string").alias("last_date"), "missing_days_in_span"),
     "prices", "11_shortest_series", rows=15)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[prices] 08_extreme_moves_summary  ->  /corpus/raw/_manifests/prices/v1_2013-2017/acq_2026-10-28/profile/20261101T112713Z/08_extreme_moves_summary
+---------------------------------+--------------------------------------------------------------------------------------------------------------+
|check                            |value                                                                                                         |
+---------------------------------+--------------------------------------------------------------------------------------------------------------+
|abs_move_gt_50pct                |7                                                                                                             |
|abs_move_gt_20pct                |102                                                                                                           |
|ret_quantiles_p01_p1_p50_p99_p999|[-0.09037745879851145, -0.042878491780664274, 0.0006335128286347746, 0.042014800668

## 6. SEC inventory and SEC filings
No screenshot needed for cell 9. Cell 10 is optional: the `01_content_summary` table.

In [9]:
# [9] SEC inventory: the fixed list that defines the SEC corpus
iv_raw = spark.read.option("header", True).csv(RAW("sec_inventory"))        # everything stays STRING
print("columns:", iv_raw.columns)
save_kv(iv_raw.select([cnt(F.col(c).isNull() | (F.trim(F.col(c)) == ""), c) for c in iv_raw.columns]),
        "sec_inventory", "01_null_or_empty_per_column")
iv = (iv_raw.withColumn("filed", F.to_date("filed_at", "yyyy-MM-dd"))
            .withColumn("url_cik", F.regexp_extract("url", r"/data/(\d+)/", 1))
            .withColumn("url_acc", F.regexp_extract("url", r"/([0-9]{10}-[0-9]{2}-[0-9]{6})\.txt$", 1)))
chk = iv.agg(F.count("*").alias("rows"), F.countDistinct("accession").alias("distinct_accessions"),
    F.countDistinct("cik").alias("distinct_issuers"),
    cnt(F.col("cik").rlike(r"^\d{10}$"), "cik_kept_as_10_digit_text"),
    cnt(F.col("accession").rlike(r"^\d{10}-\d{2}-\d{6}$"), "accession_format_ok"),
    cnt(F.col("form") == "10-K", "form_10K"), cnt(F.col("form") == "10-Q", "form_10Q"),
    cnt(F.col("filed").isNull(), "unparsable_filed_date"),
    F.min("filed").cast("string").alias("first_filed"), F.max("filed").cast("string").alias("last_filed"),
    F.countDistinct("document_kind").alias("document_kinds"),
    cnt((F.col("url_cik") == F.regexp_replace("cik", "^0+", "")) & (F.col("url_acc") == F.col("accession")),
        "url_matches_cik_and_accession"))
save_kv(chk, "sec_inventory", "02_inventory_checks")
r = chk.first()
print("vs spec (311 filings, 303 issuers, 156 10-K, 155 10-Q):",
      (r.rows, r.distinct_issuers, r.form_10K, r.form_10Q) == (311, 303, 156, 155))
save(iv.groupBy(F.year("filed").alias("year"), "form").count().orderBy("year", "form"),
     "sec_inventory", "03_filings_by_year_and_form")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

columns: ['cik', 'company_name', 'form', 'filed_at', 'accession', 'document_kind', 'url']
[sec_inventory] 01_null_or_empty_per_column  ->  /corpus/raw/_manifests/sec_inventory/v1/acq_2026-10-28/profile/20261101T112713Z/01_null_or_empty_per_column
+-------------+-----+
|check        |value|
+-------------+-----+
|cik          |0    |
|company_name |0    |
|form         |0    |
|filed_at     |0    |
|accession    |0    |
|document_kind|0    |
|url          |0    |
+-------------+-----+

[sec_inventory] 02_inventory_checks  ->  /corpus/raw/_manifests/sec_inventory/v1/acq_2026-10-28/profile/20261101T112713Z/02_inventory_checks
+-----------------------------+----------+
|check                        |value     |
+-----------------------------+----------+
|rows                         |311       |
|distinct_accessions          |311       |
|distinct_issuers             |303       |
|cik_kept_as_10_digit_text    |311       |
|accession_format_ok          |311       |
|form_10K                

In [10]:
# [10] SEC filings: content check of all 311 files, run in parallel on the executors   (optional SCREENSHOT: 01_content_summary)
import re
SEC_RAW = RAW("sec_edgar")
_HDR = {"accession": re.compile(rb"^ACCESSION NUMBER:\s*(\S+)", re.M),
        "form": re.compile(rb"^CONFORMED SUBMISSION TYPE:\s*(\S+)", re.M),
        "filed": re.compile(rb"^FILED AS OF DATE:\s*(\d{8})", re.M),
        "doc_count": re.compile(rb"^PUBLIC DOCUMENT COUNT:\s*(\d+)", re.M),
        "accepted": re.compile(rb"<ACCEPTANCE-DATETIME>(\d{14})")}
_CIK = re.compile(rb"CENTRAL INDEX KEY:\s*(\d+)")
_TYPE = re.compile(rb"^<TYPE>([^\r\n]+)", re.M)
_UUE = re.compile(rb"^begin \d{3} ", re.M)
_ERR = re.compile(rb"Request Rate Threshold Exceeded|Undeclared Automated Tool|<title>[^<]*Not Found", re.I)

inv_rows = spark.read.option("header", True).csv(RAW("sec_inventory")).select("cik", "accession", "form", "filed_at").collect()
inv_map = {f"{r.cik}_{r.accession}_{r.form}.txt": (r.cik, r.accession, r.form, r.filed_at) for r in inv_rows}
bc_inv = sc.broadcast(inv_map)

def sec_check(kv):
    path, b = kv
    name = path.rsplit("/", 1)[-1]
    exp = bc_inv.value.get(name)
    e_cik, e_acc, e_form, e_filed = exp if exp else ("", "", "", "")
    head = b[:200000]
    g = {k: (m.group(1).decode("ascii", "replace") if (m := rx.search(head)) else "") for k, rx in _HDR.items()}
    ciks = sorted({c.decode().zfill(10) for c in _CIK.findall(head.split(b"</SEC-HEADER>")[0])})
    types = [x.strip().decode("ascii", "replace") for x in _TYPE.findall(b)]
    blocks = b.count(b"\n<DOCUMENT>")
    try:
        b.decode("utf-8"); bad_utf8 = False
    except UnicodeDecodeError:
        bad_utf8 = True
    acc, f = g["accepted"], g["filed"]
    filed_iso = f"{f[:4]}-{f[4:6]}-{f[6:8]}" if f else ""
    return (name, len(b), exp is not None, b.startswith(b"<SEC-DOCUMENT>"),
            b[-4096:].rstrip().endswith(b"</SEC-DOCUMENT>"), b"</SEC-HEADER>" in head,
            g["accession"] == e_acc, g["form"] == e_form, filed_iso == e_filed, e_cik in ciks, len(ciks) > 1,
            acc[:8] if acc else "", acc[8:12] if acc else "", (acc[8:12] >= "1600") if acc else False,
            int(g["doc_count"]) if g["doc_count"] else -1, blocks, str(blocks) == g["doc_count"],
            e_form in types, "EX-101.INS" in types, (b"<ix:header" in b) or (b"<ix:nonFraction" in b),
            len(_UUE.findall(b)), bad_utf8, bool(len(b) < 65536 and _ERR.search(b)))

SEC_SCHEMA = ("file string, bytes long, in_inventory boolean, starts_ok boolean, ends_ok boolean, header_closed boolean, "
              "accession_match boolean, form_match boolean, filed_match boolean, cik_match boolean, multi_filer boolean, "
              "accepted_date string, accepted_hhmm string, accepted_after_close boolean, hdr_doc_count int, doc_blocks int, "
              "doc_count_match boolean, main_doc_present boolean, xbrl_instance boolean, inline_xbrl boolean, "
              "uuencoded_docs int, non_utf8 boolean, error_page boolean")
with timed("sec_edgar_content_check"):
    sec_df = spark.createDataFrame(sc.binaryFiles(SEC_RAW, minPartitions=48).map(sec_check), SEC_SCHEMA).cache()
    n_files = sec_df.count()

save_kv(sec_df.agg(F.count("*").alias("files"), F.sum("bytes").alias("bytes"),
    F.min("bytes").alias("min_bytes"), F.expr("percentile_approx(bytes, 0.5)").alias("median_bytes"), F.max("bytes").alias("max_bytes"),
    cnt(F.col("in_inventory"), "files_in_inventory"), cnt(F.col("starts_ok"), "starts_with_SEC_DOCUMENT"),
    cnt(F.col("ends_ok"), "ends_with_SEC_DOCUMENT"), cnt(F.col("header_closed"), "header_closed"),
    cnt(F.col("accession_match"), "header_accession_matches_inventory"), cnt(F.col("form_match"), "header_form_matches"),
    cnt(F.col("filed_match"), "header_filing_date_matches"), cnt(F.col("cik_match"), "inventory_cik_in_header"),
    cnt(F.col("main_doc_present"), "main_10K_or_10Q_document_present"),
    cnt(F.col("doc_count_match"), "public_doc_count_equals_blocks"),
    cnt(F.col("xbrl_instance") | F.col("inline_xbrl"), "has_xbrl"), cnt(~(F.col("xbrl_instance") | F.col("inline_xbrl")), "no_xbrl"),
    cnt(F.col("uuencoded_docs") > 0, "files_with_uuencoded_binaries"), cnt(F.col("multi_filer"), "multi_filer_headers"),
    cnt(F.col("non_utf8"), "not_valid_utf8"), cnt(F.col("error_page"), "error_pages_served_as_200")),
    "sec_edgar", "01_content_summary")
save(sec_df.groupBy(F.when(F.col("accepted_hhmm") < "0930", "before 09:30 ET")
                     .when(F.col("accepted_hhmm") < "1600", "09:30-16:00 ET").otherwise("16:00 ET or later").alias("accepted"))
          .count().orderBy("accepted"), "sec_edgar", "02_acceptance_time")
missing = sorted(set(inv_map) - set(r.file for r in sec_df.select("file").collect()))
print("inventory rows without a file in HDFS:", len(missing), "| files in HDFS:", len(fs_list(SEC_RAW)))
bad = sec_df.filter(~(F.col("starts_ok") & F.col("ends_ok") & F.col("header_closed") & F.col("accession_match")
                      & F.col("form_match") & F.col("filed_match") & F.col("cik_match") & F.col("main_doc_present"))
                    | F.col("error_page") | F.col("non_utf8"))
save(bad.select("file", "starts_ok", "ends_ok", "accession_match", "form_match", "filed_match", "cik_match", "error_page"),
     "sec_edgar", "03_files_with_problems")
save(sec_df.orderBy("file"), "sec_edgar", "04_per_file_checks", rows=5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

-- sec_edgar_content_check: 18.4 s
[sec_edgar] 01_content_summary  ->  /corpus/raw/_manifests/sec_edgar/v1/acq_2026-10-28/profile/20261101T112713Z/01_content_summary
+----------------------------------+----------+
|check                             |value     |
+----------------------------------+----------+
|files                             |311       |
|bytes                             |2777408405|
|min_bytes                         |77364     |
|median_bytes                      |5268992   |
|max_bytes                         |127913308 |
|files_in_inventory                |311       |
|starts_with_SEC_DOCUMENT          |311       |
|ends_with_SEC_DOCUMENT            |311       |
|header_closed                     |311       |
|header_accession_matches_inventory|311       |
|header_form_matches               |311       |
|header_filing_date_matches        |311       |
|inventory_cik_in_header           |311       |
|main_10K_or_10Q_document_present  |311       |
|public_doc_count_

## 7. GDELT
Cell 11 scans the zips inside Spark (nothing is expanded on HDFS). With `FULL_RUN = False` it scans 5 files.
**Screenshot (full run only):** cell 12, the `07_reconciliation_to_spec` table.

In [17]:
FULL_RUN = True; GDELT_LIMIT = None; WIKI_LINE_LIMIT = None
print("FULL_RUN =", FULL_RUN, "| GDELT_LIMIT =", GDELT_LIMIT, "| WIKI_LINE_LIMIT =", WIKI_LINE_LIMIT)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

FULL_RUN = True | GDELT_LIMIT = None | WIKI_LINE_LIMIT = None

In [18]:
# [11] GDELT: read the zips inside Spark (nothing is expanded on HDFS) and measure every file
import zipfile, io, hashlib, datetime as dt
GDELT_RAW = RAW("gdelt")
W0, W1 = b"20150701", b"20170630"                       # spec window: July 2015 - June 2017

_ORD, _d = {}, dt.date(1979, 1, 1)                      # valid YYYYMMDD -> day number (exact calendar check)
while _d <= dt.date(2030, 12, 31):
    _ORD[_d.strftime("%Y%m%d").encode()] = _d.toordinal(); _d += dt.timedelta(days=1)

md5_pub = {}                                            # publisher's md5 list, copied to HDFS in cell 2
for ln in sc.textFile(f"{MAN('gdelt')}/gdelt_md5sums.txt").collect():
    x = ln.split()
    if len(x) == 2: md5_pub[x[1]] = x[0]
bc_md5 = sc.broadcast(md5_pub)

def _bucket(d):
    return ("event_after_file_date" if d < 0 else "same_day" if d == 0 else "1_day" if d == 1 else "2_7_days" if d <= 7
            else "8_30_days" if d <= 30 else "31_365_days" if d <= 365 else "over_1_year")

def gdelt_file(kv):
    path, data = kv
    name = path.rsplit("/", 1)[-1]; fdate = name[:8]; fdate_b = fdate.encode(); f_ord = _ORD.get(fdate_b, 0)
    md5_ok = hashlib.md5(data).hexdigest() == bc_md5.value.get(name, "-")
    fc, lag, ysql, seen = {}, {}, {}, set()
    n = n58 = bad_id = dup = bad_date = nonutf8 = r2 = r3 = r4 = 0
    n_a1 = n_a2 = n_ev = n_geo = n_url = n_tone = bad_num = bad_quad = bad_geo = da_ne = yr_ne = 0
    lo = hi = None; members = expanded = 0; err = ""; zerr = False
    try:
        zf = zipfile.ZipFile(io.BytesIO(data)); infos = zf.infolist(); members = len(infos)
        if members:
            expanded = infos[0].file_size
            with zf.open(infos[0]) as fh:
                for line in fh:
                    n += 1
                    parts = line.rstrip(b"\r\n").split(b"\t"); k = str(len(parts))
                    fc[k] = fc.get(k, 0) + 1
                    try: line.decode("utf-8")
                    except UnicodeDecodeError: nonutf8 += 1
                    if len(parts) != 58: continue
                    n58 += 1
                    eid, sd = parts[0], parts[1]
                    id_ok, is_dup = eid.isdigit(), False
                    if id_ok:
                        if eid in seen: dup += 1; is_dup = True
                        else: seen.add(eid)
                        v = int(eid)
                        if lo is None or v < lo: lo = v
                        if hi is None or v > hi: hi = v
                    else:
                        bad_id += 1
                    o = _ORD.get(sd)
                    if o is None:
                        bad_date += 1
                    else:
                        bk = _bucket(f_ord - o); lag[bk] = lag.get(bk, 0) + 1
                        y = sd[:4].decode(); ysql[y] = ysql.get(y, 0) + 1
                        if parts[3] != sd[:4]: yr_ne += 1
                    if id_ok and o is not None:
                        r2 += 1
                        if not is_dup:
                            r3 += 1
                            if W0 <= sd <= W1: r4 += 1
                    if parts[56] != fdate_b: da_ne += 1
                    if not parts[5]: n_a1 += 1
                    if not parts[15]: n_a2 += 1
                    if not parts[26]: n_ev += 1
                    if not parts[51]: n_geo += 1
                    if not parts[57]: n_url += 1
                    if not parts[34]: n_tone += 1
                    try:
                        float(parts[30]); float(parts[31]); float(parts[34])
                    except ValueError:
                        bad_num += 1
                    if parts[29] not in (b"1", b"2", b"3", b"4"): bad_quad += 1
                    if parts[53] or parts[54]:
                        try:
                            if not (-90 <= float(parts[53]) <= 90 and -180 <= float(parts[54]) <= 180): bad_geo += 1
                        except ValueError:
                            bad_geo += 1
    except Exception as e:
        zerr = True; err = repr(e)[:150]
    return (name, fdate, len(data), expanded, md5_ok, members, zerr, err, n, n58, bad_id, dup, bad_date, nonutf8,
            n_a1, n_a2, n_ev, n_geo, n_url, n_tone, bad_num, bad_quad, bad_geo, da_ne, yr_ne, r2, r3, r4, lo, hi, fc, lag, ysql)

GDELT_SCHEMA = ("file string, file_date string, zip_bytes long, expanded_bytes long, md5_matches_publisher boolean, "
    "members int, zip_error boolean, error string, rows long, rows_58_fields long, bad_event_id long, duplicate_event_id long, "
    "bad_sqldate long, non_utf8_lines long, null_actor1 long, null_actor2 long, null_eventcode long, null_actiongeo_country long, "
    "null_sourceurl long, null_avgtone long, bad_numeric long, bad_quadclass long, bad_geo long, dateadded_ne_filedate long, "
    "year_ne_sqldate long, rule_r2 long, rule_r3 long, rule_r4 long, min_event_id long, max_event_id long, "
    "field_count_hist map<string,bigint>, sqldate_lag map<string,bigint>, sqldate_year map<string,bigint>")

paths = sorted(p_ for p_, _, _ in fs_list(GDELT_RAW) if p_.endswith(".zip"))
if GDELT_LIMIT: paths = paths[:GDELT_LIMIT]
print(f"GDELT files to scan: {len(paths)} ({'DRY RUN' if GDELT_LIMIT else 'FULL RUN'})")
with timed("gdelt_scan"):
    g = spark.createDataFrame(sc.binaryFiles(",".join(paths), minPartitions=min(len(paths), 256)).map(gdelt_file),
                              GDELT_SCHEMA).cache()
    print("files scanned:", g.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

GDELT files to scan: 731 (FULL RUN)
files scanned: 731
-- gdelt_scan: 349.8 s

In [19]:
# [12] GDELT results   (SCREENSHOT, full run only: 07_reconciliation_to_spec)
save_kv(g.agg(F.count("*").alias("files"), F.countDistinct("file_date").alias("distinct_file_dates"),
    F.min("file_date").alias("first_file_date"), F.max("file_date").alias("last_file_date"),
    F.sum("zip_bytes").alias("zip_bytes"), F.sum("expanded_bytes").alias("expanded_bytes"),
    cnt(F.col("md5_matches_publisher"), "md5_matches_publisher"), cnt(F.col("zip_error"), "zip_or_crc_errors"),
    cnt(F.col("members") != 1, "zips_without_exactly_one_csv"), F.sum("rows").alias("rows_total"),
    F.min("rows").alias("min_rows_per_day"), F.expr("percentile_approx(rows, 0.5)").alias("median_rows_per_day"),
    F.max("rows").alias("max_rows_per_day")), "gdelt", "01_files_and_bytes")
tb = g.agg(F.sum("zip_bytes"), F.sum("expanded_bytes")).first()
print(f"zip {tb[0]/1e9:.2f} GB (spec 9.61) | expanded {tb[1]/1e9:.2f} GB (spec 54.71)  [decimal GB]")

exp_days = {(dt.date(2015, 7, 1) + dt.timedelta(days=i)).strftime("%Y%m%d") for i in range((dt.date(2017, 6, 30) - dt.date(2015, 7, 1)).days + 1)}
have = {r.file_date for r in g.select("file_date").collect()}
print(f"days in Jul-2015..Jun-2017: {len(exp_days)} | files for those days: {len(have & exp_days)} | "
      f"missing: {len(exp_days - have)} | outside window: {len(have - exp_days)}"
      + ("   (dry run: ignore 'missing')" if GDELT_LIMIT else ""))

save(g.select(F.explode("field_count_hist").alias("fields_per_line", "lines")).groupBy("fields_per_line")
      .agg(F.sum("lines").alias("lines")).orderBy("fields_per_line"), "gdelt", "02_field_count_histogram")
save_kv(g.agg(F.sum("rows_58_fields").alias("rows_with_58_fields"), F.sum("bad_event_id").alias("bad_event_id"),
    F.sum("duplicate_event_id").alias("duplicate_event_id_within_file"), F.sum("bad_sqldate").alias("invalid_sqldate"),
    F.sum("non_utf8_lines").alias("non_utf8_lines"), F.sum("null_actor1").alias("empty_actor1code"),
    F.sum("null_actor2").alias("empty_actor2code"), F.sum("null_eventcode").alias("empty_eventcode"),
    F.sum("null_actiongeo_country").alias("empty_actiongeo_country"), F.sum("null_sourceurl").alias("empty_sourceurl"),
    F.sum("null_avgtone").alias("empty_avgtone"), F.sum("bad_numeric").alias("non_numeric_goldstein_mentions_tone"),
    F.sum("bad_quadclass").alias("quadclass_not_1_to_4"), F.sum("bad_geo").alias("actiongeo_lat_long_out_of_range"),
    F.sum("dateadded_ne_filedate").alias("dateadded_differs_from_file_date"),
    F.sum("year_ne_sqldate").alias("year_field_differs_from_sqldate")), "gdelt", "03_quality_counts")
save(g.select(F.explode("sqldate_lag").alias("event_date_vs_file_date", "events")).groupBy("event_date_vs_file_date")
      .agg(F.sum("events").alias("events")).orderBy(F.desc("events")), "gdelt", "04_event_date_lag")
save(g.select(F.explode("sqldate_year").alias("sqldate_year", "events")).groupBy("sqldate_year")
      .agg(F.sum("events").alias("events")).orderBy("sqldate_year"), "gdelt", "05_events_by_sqldate_year", rows=60)
ov = (g.select("file_date", "min_event_id", "max_event_id")
       .withColumn("prev_max_id", F.lag("max_event_id").over(W.orderBy("file_date"))))
save_kv(ov.agg(cnt(F.col("min_event_id") <= F.col("prev_max_id"), "files_whose_event_id_range_overlaps_previous_file")),
        "gdelt", "06_event_id_range_overlap")

rules = g.agg(F.sum("rows").alias("R0_all_lines"), F.sum("rows_58_fields").alias("R1_58_fields"),
              F.sum("rule_r2").alias("R2_plus_valid_id_and_sqldate"), F.sum("rule_r3").alias("R3_plus_no_duplicate_id"),
              F.sum("rule_r4").alias("R4_plus_sqldate_in_Jul2015_Jun2017")).first().asDict()
save(spark.createDataFrame([(k, v, v - 144200000, round(100.0 * (v - 144200000) / 144200000, 3)) for k, v in rules.items()],
     "rule string, events long, diff_vs_spec_144_2M long, diff_pct double"), "gdelt", "07_reconciliation_to_spec")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[gdelt] 01_files_and_bytes  ->  /corpus/raw/_manifests/gdelt/v1_events_2015-2017/acq_2026-10-29/profile/20261101T112713Z/01_files_and_bytes
+----------------------------+-----------+
|check                       |value      |
+----------------------------+-----------+
|files                       |731        |
|distinct_file_dates         |731        |
|first_file_date             |20150701   |
|last_file_date              |20170630   |
|zip_bytes                   |9611684528 |
|expanded_bytes              |54706262609|
|md5_matches_publisher       |731        |
|zip_or_crc_errors           |0          |
|zips_without_exactly_one_csv|0          |
|rows_total                  |144207180  |
|min_rows_per_day            |29167      |
|median_rows_per_day         |214138     |
|max_rows_per_day            |277014     |
+----------------------------+-----------+

zip 9.61 GB (spec 9.61) | expanded 54.71 GB (spec 54.71)  [decimal GB]
days in Jul-2015..Jun-2017: 731 | files for those days: 7

## 8. Wikinews
Hadoop reads the `.bz2` dumps directly; one task per file keeps the XML in order. With `FULL_RUN = False` only the first 3M lines of each file are read.

In [30]:
# [13] Wikinews: Hadoop reads the .bz2 dumps directly; Python parses the MediaWiki XML line by line
import re
from itertools import islice
WIKI_RAW = RAW("wikinews")
wiki_files = [(wp, wn) for wp, wn, _ in fs_list(WIKI_RAW) if wp.endswith(".bz2")]
print("files:", [(wp.rsplit("/", 1)[-1], wn) for wp, wn in wiki_files],
      "| lines per file:", WIKI_LINE_LIMIT or "all (full run)")
_BYTES = re.compile(r'bytes="(\d+)"')

def _between(ln, start, end):
    i = ln.find(end, start)
    return ln[start:i] if i >= 0 else None
def _toint(s):
    try: return int(s)
    except (TypeError, ValueError): return None

def wiki_parse(fid, it):
    page_id = ns = None; rev = None
    for ln in it:
        if ln.startswith("    "):
            if ln.startswith("    <ns>"):
                ns = _toint(_between(ln, 8, "</ns>"))
            elif ln.startswith("    <id>"):
                page_id = _toint(_between(ln, 8, "</id>"))
                yield ("P", fid, page_id, ns, None, None, None, None, None, None)
            elif ln.startswith("    <revision>"):
                rev = [None, None, False, False, None]       # id, timestamp, contributor deleted, text deleted, text bytes
            elif ln.startswith("    </revision>"):
                if rev is None: yield ("O", fid, page_id, ns, None, None, None, None, None, None)
                else:
                    yield ("R", fid, page_id, ns, rev[0], rev[1], rev[2], rev[3], rev[4], None); rev = None
            elif ln.startswith("    <dbname>") or ln.startswith("    <generator>"):
                yield ("H", fid, None, None, None, None, None, None, None, ln.strip())
            elif rev is not None and ln.startswith("      <"):
                if ln.startswith("      <id>"): rev[0] = _toint(_between(ln, 10, "</id>"))
                elif ln.startswith("      <timestamp>"): rev[1] = _between(ln, 17, "</timestamp>")
                elif ln.startswith("      <contributor"): rev[2] = 'deleted="deleted"' in ln
                elif ln.startswith("      <text"):
                    rev[3] = 'deleted="deleted"' in ln
                    m = _BYTES.search(ln); rev[4] = int(m.group(1)) if m else None
        elif ln.startswith("  <page>"):
            page_id = ns = None; rev = None
        elif ln.startswith("<mediawiki"):
            yield ("H", fid, None, None, None, None, None, None, None, ln[:140])
        elif ln.startswith("</mediawiki>"):
            yield ("X", fid, None, None, None, None, None, None, None, None)
    if rev is not None: yield ("T", fid, page_id, ns, None, None, None, None, None, None)

rdds = []
for fid, (wp, wn) in enumerate(wiki_files):
    rr = sc.newAPIHadoopFile(wp, "org.apache.hadoop.mapreduce.lib.input.TextInputFormat",
            "org.apache.hadoop.io.LongWritable", "org.apache.hadoop.io.Text",
            conf={"mapreduce.input.fileinputformat.split.minsize": "100000000000"})   # one split per file = exact, in order
    rdds.append(rr.values().mapPartitions(
        lambda it, fid=fid: wiki_parse(fid, islice(it, WIKI_LINE_LIMIT) if WIKI_LINE_LIMIT else it)))
WIKI_SCHEMA = ("kind string, file_id int, page_id long, ns int, rev_id long, ts string, contributor_missing boolean, "
               "text_missing boolean, text_bytes long, info string")
with timed("wikinews_scan"):
    wk = spark.createDataFrame(sc.union(rdds), WIKI_SCHEMA).cache()
    print("parsed records:", wk.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

files: [('enwikinews-20261001-pages-meta-history1.xml-p1p481753.bz2', 482143381), ('enwikinews-20261001-pages-meta-history1.xml-p481754p3096188.bz2', 179859131)] | lines per file: all (full run)
parsed records: 2220138
-- wikinews_scan: 648.6 s

In [31]:
# [14] Wikinews results
files_df = spark.createDataFrame([(i, wp.rsplit("/", 1)[-1], wn) for i, (wp, wn) in enumerate(wiki_files)],
                                 "file_id int, file string, compressed_bytes long")
save(wk.groupBy("file_id").agg(cnt(F.col("kind") == "P", "pages"), cnt(F.col("kind") == "R", "revisions"),
        cnt(F.col("kind") == "O", "orphan_revision_ends"), cnt(F.col("kind") == "T", "truncated_revisions"),
        cnt(F.col("kind") == "X", "closing_mediawiki_tag"))
      .join(files_df, "file_id").orderBy("file_id").select("file", "compressed_bytes", "pages", "revisions",
        "orphan_revision_ends", "truncated_revisions", "closing_mediawiki_tag"), "wikinews", "01_per_file")
save(wk.filter("kind = 'H'").select("file_id", "info").distinct().orderBy("file_id", "info"), "wikinews", "02_dump_header", 12)

rev, pg = wk.filter("kind = 'R'"), wk.filter("kind = 'P'")
ts_ok = F.col("ts").rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$")
save_kv(rev.agg(F.count("*").alias("revisions"), F.countDistinct("rev_id").alias("distinct_revision_ids"),
    cnt(F.col("rev_id").isNull(), "missing_revision_id"), cnt(F.col("page_id").isNull(), "revisions_without_page"),
    cnt(~ts_ok | F.col("ts").isNull(), "invalid_or_missing_timestamp"),
    F.min("ts").alias("first_timestamp"), F.max("ts").alias("last_timestamp"),
    cnt(F.col("contributor_missing"), "contributor_deleted"), cnt(F.col("text_missing"), "text_deleted"),
    cnt(F.col("text_bytes").isNull(), "text_bytes_unknown"), cnt(F.col("text_bytes") == 0, "empty_text")),
    "wikinews", "03_revision_quality")
save_kv(pg.agg(F.count("*").alias("page_records"), F.countDistinct("page_id").alias("distinct_pages"),
               cnt(F.col("ns").isNull(), "pages_without_namespace")), "wikinews", "04_page_quality")
save(rev.groupBy("ns").agg(F.count("*").alias("revisions")).orderBy(F.desc("revisions")),
     "wikinews", "05_revisions_by_namespace", rows=40)
save(rev.groupBy(F.substring("ts", 1, 4).alias("year")).agg(F.count("*").alias("revisions"),
        cnt(F.col("ns") == 0, "article_revisions_ns0")).orderBy("year"), "wikinews", "06_revisions_by_year", rows=40)
win = (F.col("ts") >= "2013-01-01") & (F.col("ts") < "2018-01-01")
w = rev.agg(F.count("*").alias("all_revisions"), cnt(win, "revisions_2013_2017"),
            cnt(win & (F.col("ns") == 0), "article_revisions_2013_2017"),
            F.countDistinct(F.when(win & (F.col("ns") == 0), F.col("page_id"))).alias("articles_with_revision_2013_2017"))
save_kv(w, "wikinews", "07_scope_2013_2017")
wr = w.first()
if wr.all_revisions: print(f"share of all revisions in 2013-2017: {100.0 * wr.revisions_2013_2017 / wr.all_revisions:.1f}%")
save_kv(rev.groupBy("page_id").count().agg(F.count("*").alias("pages_with_revisions"),
        F.expr("percentile_approx(count, array(0.5,0.9,0.99))").alias("revisions_per_page_p50_p90_p99"),
        F.max("count").alias("max_revisions_on_one_page")), "wikinews", "08_revisions_per_page")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[wikinews] 01_per_file  ->  /corpus/raw/_manifests/wikinews/20261001_meta-history/acq_2026-10-29/profile/20261101T112713Z/01_per_file
+---------------------------------------------------------------+----------------+------+---------+--------------------+-------------------+---------------------+
|file                                                           |compressed_bytes|pages |revisions|orphan_revision_ends|truncated_revisions|closing_mediawiki_tag|
+---------------------------------------------------------------+----------------+------+---------+--------------------+-------------------+---------------------+
|enwikinews-20261001-pages-meta-history1.xml-p1p481753.bz2      |482143381       |142719|1500001  |0                   |0                  |1                    |
|enwikinews-20261001-pages-meta-history1.xml-p481754p3096188.bz2|179859131       |123181|454229   |0                   |0                  |1                    |
+--------------------------------------------------

## 9. Cross-source summary
**Screenshot:** the `09_relationships` table (which links between sources are supported).

In [32]:
# [15] Cross-source summary: date coverage, supported relationships, timings   (SCREENSHOT: 09_relationships)
iso = lambda s: f"{s[:4]}-{s[4:6]}-{s[6:8]}" if s else None
pr = p.agg(F.min("date").cast("string"), F.max("date").cast("string"), F.countDistinct("series")).first()
sa = sec_df.agg(F.min("accepted_date"), F.max("accepted_date"), cnt(F.col("accession_match") & F.col("cik_match"), "ok")).first()
gd = g.agg(F.min("file_date"), F.max("file_date"), F.sum("rows")).first()
wr2 = rev.agg(F.min("ts"), F.max("ts"), F.count("*")).first()
cov = [("prices", "trading date (ET)", pr[0], pr[1], f"{pr[2]} tickers"),
       ("sec_edgar", "acceptance date (ET)", iso(sa[0]), iso(sa[1]), "311 filings"),
       ("gdelt", "file date = DATEADDED", iso(gd[0]), iso(gd[1]), f"{gd[2]:,} lines"),
       ("wikinews", "revision timestamp (UTC)", wr2[0], wr2[1], f"{wr2[2]:,} revisions")]
save(spark.createDataFrame(cov, "source string, time_field string, first string, last string, size string"),
     "_run", "08_date_coverage")
trade_days = p.select(F.date_format("date", "yyyyMMdd").alias("d")).distinct()
gd_days = g.select(F.col("file_date").alias("d")).distinct()
n_overlap = trade_days.join(gd_days, "d").count()
rel = [("sec_inventory -> sec_edgar", "accession + CIK + form", "SUPPORTED", f"{sa[2]}/311 headers match the inventory"),
       ("sec_edgar -> prices", "none (no ticker in either)", "NOT SUPPORTED", "inventory has no ticker; documented limitation"),
       ("prices <-> gdelt", "date only", "BY DATE", f"{n_overlap} trading days have a GDELT file; no ticker in GDELT"),
       ("prices <-> wikinews", "date only", "BY DATE", "revision timestamps vs trading dates; no ticker in Wikinews"),
       ("sec_edgar <-> prices", "acceptance date -> next trading day", "BY DATE", "availability rule, not an entity link")]
save(spark.createDataFrame(rel, "relationship string, key string, status string, evidence string"), "_run", "09_relationships")
save(spark.createDataFrame([(k, float(v)) for k, v in TIMINGS.items()], "step string, seconds double"), "_run", "10_timings")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[_run] 08_date_coverage  ->  /corpus/raw/_manifests/_task1_run/20261101T112713Z/08_date_coverage
+---------+------------------------+--------------------+--------------------+-------------------+
|source   |time_field              |first               |last                |size               |
+---------+------------------------+--------------------+--------------------+-------------------+
|prices   |trading date (ET)       |2013-02-08          |2018-02-07          |505 tickers        |
|sec_edgar|acceptance date (ET)    |2013-01-14          |2017-12-20          |311 filings        |
|gdelt    |file date = DATEADDED   |2015-07-01          |2017-06-30          |144,207,180 lines  |
|wikinews |revision timestamp (UTC)|2004-11-08T08:11:42Z|2026-09-30T13:50:02Z|1,954,230 revisions|
+---------+------------------------+--------------------+--------------------+-------------------+

[_run] 09_relationships  ->  /corpus/raw/_manifests/_task1_run/20261101T112713Z/09_relationships
+------------

## 10. Save the evidence to S3
Run after a **full** run (`FULL_RUN = True`). Then export this notebook (File > Download as > Notebook) for the submission.

In [34]:
# [16] Copy all results to S3 so they survive the lab session
FileUtil = _jvm.org.apache.hadoop.fs.FileUtil
EV_S3 = f"s3://{S3_BUCKET}/evidence/task1_{RUN_ID}" + ("" if FULL_RUN else "_dryrun")
done = []
for s in list(SRC) + ["_run"]:
    d = out_dir(s)
    if not fs_exists(d): continue
    dst = f"{EV_S3}/{s}"
    ok = FileUtil.copy(_fs(d), _Path(d), _fs(dst), _Path(dst), False, True, _hconf)
    done.append((s, d, dst, "copied" if ok else "FAILED"))
info = [f"run_id={RUN_ID}", f"application_id={sc.applicationId}", f"spark={spark.version}", f"full_run={FULL_RUN}"] \
       + [f"timing[{k}]={v}s" for k, v in TIMINGS.items()]
info_path = f"{EV_S3}/run_info"
if fs_exists(info_path): _fs(info_path).delete(_Path(info_path), True)
sc.parallelize(info, 1).saveAsTextFile(info_path)
spark.createDataFrame(done, "source string, from_hdfs string, to_s3 string, result string").show(20, truncate=False)
print("evidence saved:", EV_S3, "|", len(fs_list(EV_S3)), "files")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------+---------------------------------------------------------------------------------------------+------------------------------------------------------------------+------+
|source       |from_hdfs                                                                                    |to_s3                                                             |result|
+-------------+---------------------------------------------------------------------------------------------+------------------------------------------------------------------+------+
|prices       |/corpus/raw/_manifests/prices/v1_2013-2017/acq_2026-10-28/profile/20261101T112713Z           |s3://inf2006-bigdata/evidence/task1_20261101T112713Z/prices       |copied|
|sec_inventory|/corpus/raw/_manifests/sec_inventory/v1/acq_2026-10-28/profile/20261101T112713Z              |s3://inf2006-bigdata/evidence/task1_20261101T112713Z/sec_inventory|copied|
|sec_edgar    |/corpus/raw/_manifests/sec_edgar/v1/acq_2026-10-28/profile/202611